In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import multiprocessing
import os
import pickle

try:
    cores = int(os.environ["SLURM_JOB_CPUS_PER_NODE"])
except:
    cores = multiprocessing.cpu_count()

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import jax
import jax.numpy as jnp
import numpy as np
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.bijectors import Log
from tensorflow_probability.substrates.jax.distributions import (
    Normal,
)

from gallifrey.inference import log_likelihood_function
from gallifrey.kernelsearch import print_kernel_summary, get_trainables
from gallifrey.mcmc import nuts_warmup, run_mcmc
from gallifrey.util import calculate_example_lightcurve

rng_key = jax.random.PRNGKey(42)

## LOAD DATA AND MODEL

In [3]:
white_noise_std = 0.001
phi = 0
(
    t_train,
    lc_train,
    train_mask,
    t,
    lightcurve,
    systematics,
    noise,
    mask,
) = calculate_example_lightcurve(noise_std=white_noise_std, phi=phi)

noise_std = white_noise_std / jnp.sqrt(1 - phi**2)

model_name = "gpmodel_periodic_kernel"
with open(f"../data/processed/toy_data/gp_models/{model_name}", "rb") as file:
    model = pickle.load(file)

print_kernel_summary(model)

Kernel Summary

Kernel Structure: Periodic

--------------------------------------------------------------------------------
Kernel               Property             Value                Trainable 
--------------------------------------------------------------------------------
Periodic            lengthscale          0.1332483            True      

                    variance             8.578458e-06         True      

                    period               1.3515452            True      
--------------------------------------------------------------------------------


## CREATE LIGHT CURVE MODEL

In [4]:
@jit
def lc_model(t, params):
    # The light curve calculation requires an orbit
    orbit = orbits.keplerian.Body(
        period=15,
        radius=params[0],
        inclination=jnp.deg2rad(89),
        time_transit=0,
    )

    lc = LimbDarkLightCurve([params[1], params[2]]).light_curve(orbit, t=t)
    return lc

## FIT LC

In [5]:
initial_position = {
    "gp_parameter": get_trainables(
        model, unconstrain=True
    ),  # gp is fitted in unconstrained space, so set initial value to that
    "lc_parameter": jnp.asarray([0.2, 0.2, 0.5]),
}

param_priors = {
    "gp_parameter": Normal(loc=initial_position["gp_parameter"], scale=1),
    "lc_parameter": Normal(
        loc=initial_position["lc_parameter"],
        scale=[0.1, 0.3, 0.2],
    ),
}

In [6]:
log_likelihood = log_likelihood_function(
    model,
    lc_model,
    t_train,
    lc_train,
    train_mask,
    fix_gp=False,
    compile=True,
)


@jit
def log_priors(params):
    gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
    lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
    return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)


@jit
def log_probability(params):
    return log_likelihood(params) + log_priors(params)


neg_log_probability = jit(lambda params: -log_probability(params))

In [7]:
lbfgsb = ScipyMinimize(fun=neg_log_probability, method="l-bfgs-b")
lbfgsb_sol = lbfgsb.run(initial_position)
print("Best fit parameter: ", lbfgsb_sol.params["lc_parameter"])

Best fit parameter:  [0.10334392 0.0469249  0.461069  ]


## RUN MCMC

In [8]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 500
num_samples = 500
num_chains = cores

rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)

state, parameters = nuts_warmup(
    warmup_key,
    log_probability,
    initial_position,
    num_steps=num_adapt,
)

initial_positions = {
    "gp_parameter": jnp.tile(state.position["gp_parameter"], (num_chains, 1)),
    "lc_parameter": jnp.tile(state.position["lc_parameter"], (num_chains, 1)),
}

final_state, state_history, info_history = run_mcmc(
    sample_key,
    log_probability,
    parameters,
    initial_positions,
    num_steps=num_samples,
)

Running window adaptation


In [10]:
np.save(
    f"../data/processed/toy_data/mcmc_chains/{model_name}_parameter.npy",
    np.array(
        state_history.position["lc_parameter"].reshape(
            -1, state_history.position["lc_parameter"].shape[-1]
        )
    ),
)
np.save(
    f"../data/processed/toy_data/mcmc_chains/{model_name}_parameter_gp.npy",
    np.array(
        state_history.position["gp_parameter"].reshape(
            -1, state_history.position["gp_parameter"].shape[-1]
        )
    ),
)